## Credentials

In [1]:
# 1. Update paths explicitly to link your downloaded security clearance file

# create a service account on google cloud console and generate json key
key_path = "google_project_key.json" 

# google cloud console project id
project_id = "duality-extended"

## Fetching Data

In [ ]:
import duckdb
import geopandas as gpd

# 1. Connect and initialize cloud drivers
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Max out processing speed on your machine
con.execute("SET threads = 8;")
con.execute("SET max_memory = '8GB';")

# 2. Get the active release version string
latest_release = con.execute("""
    SELECT latest FROM read_json('https://stac.overturemaps.org/catalog.json')
""").fetchone()[0]

print(f"Targeting active release: {latest_release}")

# 3. Your exact 1 km target box coordinates (Long, Lat)
xmin, ymin, xmax, ymax = -69.8040, 9.9660, -69.7950, 9.9750

# 4. LIGHTNING FAST QUERY: Use Overture's native numerical 'bbox' columns.
# This pushes down the filter straight to S3/Azure, reading only the exact bytes needed.
query = f"""
    SELECT 
        id, 
        geometry
    FROM read_parquet(
        'az://overturemapswestus2.blob.core.windows.net/release/{latest_release}/theme=buildings/type=building/*', 
        filename=true, 
        hive_partitioning=1
    )
    WHERE bbox.xmin BETWEEN {xmin} AND {xmax}
      AND bbox.ymin BETWEEN {ymin} AND {ymax}
"""

print("Streaming cropped data...")
df = con.execute(query).df()

if df.empty:
    print("\nNo buildings found inside this 1 km box.")
else:
    # Load raw geometry directly into GeoPandas
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
    print(f"\nSuccess! Loaded {len(gdf)} records in memory.")
    print(gdf.head())

Targeting active release: 2026-05-20.0
Streaming cropped data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
import os
import pandas as pd
import geopandas as gpd
from shapely import wkt
from google.cloud import bigquery
from google.oauth2 import service_account



# 2. Target coordinate box boundaries 
xmin, ymin, xmax, ymax = -69.8495, 9.9205, -69.7495, 10.0205

print("Connecting to live cloud infrastructure via explicit credentials...")
try:
    # Authenticate via JSON key configuration parameters
    credentials = service_account.Credentials.from_service_account_file(key_path)
    client = bigquery.Client(credentials=credentials, project=project_id)
    
    # 3. Clean, validated BigQuery SQL utilizing standard WKT geometry definitions
    # This processes everything cloud-side and evades local proxy blocks completely
    query = f"""
        SELECT 
            id,
            class,
            height,
            ST_AsText(geometry) AS wkt_string
        FROM `bigquery-public-data.overture_maps.building`
        WHERE ST_Intersects(
              geometry, 
              ST_GeogFromText('POLYGON(({xmin} {ymin}, {xmax} {ymin}, {xmax} {ymax}, {xmin} {ymax}, {xmin} {ymin}))')
          )
    """

    print("Executing serverless cloud filtering... (No local download occurring)")
    df = client.query(query).to_dataframe()
    
    if df.empty:
        print("\nQuery executed successfully, but no buildings matched those coordinates.")
        print("Verify your coordinate boundaries wrap around the targeted structures correctly.")
    else:
        # 4. Instantiate directly as a spatial map layer held strictly in temporary RAM
        df['geometry'] = df['wkt_string'].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
        
        # Clear text parsing artifacts to keep variables pristine
        gdf = gdf.drop(columns=['wkt_string'])
        
        print(f"\nSuccess! {len(gdf)} comprehensive latest building records are now held in memory.")
        print("\n--- In-Memory Data Preview ---")
        print(gdf[['id', 'class', 'height', 'geometry']].head())

except Exception as e:
    print(f"\nExecution failed: {e}")


Connecting to live cloud infrastructure via explicit credentials...
Executing serverless cloud filtering... (No local download occurring)

Execution failed: 403 Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas; reason: quotaExceeded, location: unbilled.analysis, message: Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas

Location: US
Job ID: e57ba12d-b7c5-4809-8143-6fbd99ada060



## Converting to CSV

In [5]:
import pandas as pd

# 1. Manually set the CRS to WGS84 (EPSG:4326) so GeoPandas knows it's dealing with lat/lon
gdf.crs = "EPSG:4326"

# 2. Project to a local UTM zone and calculate area in square meters
gdf_meters = gdf.to_crs(gdf.estimate_utm_crs())
gdf['area_sq_meters'] = gdf_meters.geometry.area

print("\n--- Updated In-Memory Data Table Matrix ---")
# Use only columns that actually exist in your gdf to avoid KeyErrors
display_cols = [col for col in ['id', 'area_sq_meters', 'geometry'] if col in gdf.columns]
print(gdf[display_cols].head())

# 3. Flatten shapes to pure string values for lightweight text logging
gdf['geometry_polygon_wkt'] = gdf.geometry.apply(lambda geom: geom.wkt)

# 4. Drop the active geometric object layer and save as a spreadsheet
df_csv = pd.DataFrame(gdf.drop(columns=['geometry']))

output_file = "venezuela_buildings1.csv" 
df_csv.to_csv(output_file, index=False)

print(f"Saved active data slice to disk: '{output_file}'")


--- Updated In-Memory Data Table Matrix ---
                                     id  area_sq_meters  \
0  0eac054c-c24e-4339-a2f5-daf00385de64       35.663447   
1  af11f45d-9a32-4151-8450-a4d62b1cbb86     2665.272902   
2  a21afdde-844b-4c06-9c81-37ee02ab0aab     1612.780356   
3  1631aeb9-d7aa-46a8-acc1-57ead568f223     2632.091182   
4  2508b4dd-a42d-4bba-95db-48b5c07046ad     2093.341662   

                                            geometry  
0  POLYGON ((-69.80011 9.97366, -69.80013 9.97361...  
1  POLYGON ((-69.79887 9.97207, -69.79898 9.97117...  
2  POLYGON ((-69.79861 9.97166, -69.79868 9.97113...  
3  POLYGON ((-69.79821 9.97195, -69.79833 9.97106...  
4  POLYGON ((-69.79897 9.97217, -69.79916 9.97218...  
Saved active data slice to disk: 'venezuela_buildings1.csv'


In [ ]:
gdf_meters = gdf.to_crs(gdf.estimate_utm_crs())
gdf['area_sq_meters'] = gdf_meters.geometry.area

print("\n--- Updated In-Memory Data Table Matrix ---")
print(gdf[['id', 'area_sq_meters', 'geometry']].head())


# Flatten shapes to pure string values for lightweight text logging
gdf['geometry_polygon_wkt'] = gdf.geometry.apply(lambda geom: geom.wkt)

# Exclude the active geometric object cache layer and save a text spreadsheet
df_csv = pd.DataFrame(gdf.drop(columns=['geometry']))
df_csv.to_csv("bengaluru_cloud_buildings_2026.csv", index=False)
print("Saved active data slice to disk: 'bengaluru_cloud_buildings_2026.csv'")


RuntimeError: crs must be set to estimate UTM CRS.

## Checking Version

In [17]:
# Query the BigQuery table schema catalog to extract the active data release name
version_query = """
    SELECT option_value AS release_version
    FROM `bigquery-public-data.overture_maps.INFORMATION_SCHEMA.TABLE_OPTIONS`
    WHERE table_name = 'building' AND option_name = 'labels'
"""

try:
    print("Checking active dataset snapshot version...")
    version_df = client.query(version_query).to_dataframe()
    
    if not version_df.empty:
        print("\n--- Overture Dataset Live Version ---")
        print(version_df['release_version'].iloc[0])
    else:
        print("Could not retrieve explicit label metadata.")
except Exception as e:
    print(f"Metadata query failed: {e}")


Checking active dataset snapshot version...


/Users/parthbansal/Satyukt Analytics/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



--- Overture Dataset Live Version ---
[STRUCT("release", "2026-05-20_0")]
